In [1]:
%pip install -q mediapipe

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install mediapipe opencv-python

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#===================imports=======================
import cv2
import mediapipe as mp
import numpy as np
import time

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [10]:
# ================== MODEL PATHS ==================
POSE_MODEL_PATH = "pose_landmarker_full.task"
HAND_MODEL_PATH = "hand_landmarker.task"

In [11]:
# ================== BASE OPTIONS ==================
BaseOptions = python.BaseOptions
VisionRunningMode = vision.RunningMode

In [12]:
# ================== POSE SETUP ==================
PoseLandmarker = vision.PoseLandmarker
PoseLandmarkerOptions = vision.PoseLandmarkerOptions

pose_options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=POSE_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1
)

In [13]:
# ================== HAND SETUP ==================
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions

hand_options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)

In [14]:
# ================== ANGLE CALCULATION ==================
def calculate_angle(a, b, c):
    a = np.array([a.x, a.y])
    b = np.array([b.x, b.y])
    c = np.array([c.x, c.y])

    ba = a - b
    bc = c - b

    cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.arccos(np.clip(cosine_angle, -1.0, 1.0))
    return np.degrees(angle)

In [15]:
# ================== DRAW CONNECTION FUNCTION ==================
def draw_connections(frame, landmarks, connections, w, h, color):
    for start_idx, end_idx in connections:
        start = landmarks[start_idx]
        end = landmarks[end_idx]

        x1, y1 = int(start.x * w), int(start.y * h)
        x2, y2 = int(end.x * w), int(end.y * h)

        cv2.line(frame, (x1, y1), (x2, y2), color, 2)

# ================== HAND CONNECTIONS ==================
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20)
]

# ================== POSE CONNECTIONS ==================
POSE_CONNECTIONS = [
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16),
    (11, 23), (12, 24)
]

In [16]:
# ================== CAMERA ==================
cap = cv2.VideoCapture(0)
start_time = time.time()

window_name = "Physio Assessment - Hand + Arm"
cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

baseline_open_angle = None

with PoseLandmarker.create_from_options(pose_options) as pose_landmarker, \
     HandLandmarker.create_from_options(hand_options) as hand_landmarker:

    while True:

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break

        ret, frame = cap.read()
        if not ret:
            break

        h, w, _ = frame.shape
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb_frame
        )

        timestamp_ms = int((time.time() - start_time) * 1000)

        # ================== POSE DETECTION ==================
        pose_result = pose_landmarker.detect_for_video(mp_image, timestamp_ms)

        if pose_result.pose_landmarks:
            landmarks = pose_result.pose_landmarks[0]

            draw_connections(frame, landmarks, POSE_CONNECTIONS, w, h, (0,255,0))

            # Shoulder Raise Angle (Left Side Example)
            shoulder_angle = calculate_angle(
                landmarks[23],  # hip
                landmarks[11],  # shoulder
                landmarks[13]   # elbow
            )

            cv2.putText(frame, f"Shoulder Angle: {int(shoulder_angle)} deg",
                        (30, 60),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.9, (255,255,0), 2)

        # ================== HAND DETECTION ==================
        hand_result = hand_landmarker.detect_for_video(mp_image, timestamp_ms)

        if hand_result.hand_landmarks:
            for hand_landmarks in hand_result.hand_landmarks:

                draw_connections(frame, hand_landmarks, HAND_CONNECTIONS, w, h, (255,0,0))

                # Calculate MCP Angles
                index_angle = calculate_angle(hand_landmarks[0], hand_landmarks[5], hand_landmarks[8])
                middle_angle = calculate_angle(hand_landmarks[0], hand_landmarks[9], hand_landmarks[12])
                ring_angle = calculate_angle(hand_landmarks[0], hand_landmarks[13], hand_landmarks[16])
                pinky_angle = calculate_angle(hand_landmarks[0], hand_landmarks[17], hand_landmarks[20])

                avg_angle = (index_angle + middle_angle + ring_angle + pinky_angle) / 4

                # Convert to percentage
                open_percentage = int(((avg_angle - 60) / (180 - 60)) * 100)
                open_percentage = max(0, min(100, open_percentage))
                closed_percentage = 100 - open_percentage

                # Save baseline when fully open
                if baseline_open_angle is None and open_percentage > 90:
                    baseline_open_angle = avg_angle

                cv2.putText(frame, f"Open: {open_percentage}%",
                            (30, 120),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0,255,0), 2)

                cv2.putText(frame, f"Closed: {closed_percentage}%",
                            (30, 160),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.9, (0,0,255), 2)

        cv2.imshow(window_name, frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

# ================== CLEANUP ==================
cap.release()
cv2.destroyAllWindows()

In [ ]:
%pip install -q mysql-connector-python

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import mysql.connector
from mysql.connector import Error

# ========================================
# Example function to check and update strengths
# ========================================
def update_patient_strengths(patient_user_id, shoulder=0, elbow=0, wrist=0, grip=0, external_rotation=0):
    """
    patient_user_id: the user_id of the patient in users table
    strength values are optional, will be assigned if no existing data
    """
    connection = None
    cursor = None

    try:
        # Connect to DB
        connection = mysql.connector.connect(
            host='localhost',
            user='root',
            password='your_password',
            database='rehabilitation_system'
        )
        cursor = connection.cursor(dictionary=True)

        # Get patient record
        cursor.execute("SELECT * FROM patients WHERE user_id = %s", (patient_user_id,))
        patient = cursor.fetchone()

        if not patient:
            print("❌ No patient found with this user_id.")
            return

        # Check if any strength data exists
        has_data = any([
            patient['shoulder_strength'],
            patient['elbow_strength'],
            patient['wrist_strength'],
            patient['grip_strength'],
            patient['external_rotation']
        ])

        if not has_data:
            # If no data, assign values from MediaPipe (passed as function args)
            update_query = """
                UPDATE patients
                SET shoulder_strength = %s,
                    elbow_strength = %s,
                    wrist_strength = %s,
                    grip_strength = %s,
                    external_rotation = %s
                WHERE user_id = %s
            """
            cursor.execute(update_query, (
                shoulder, elbow, wrist, grip, external_rotation, patient_user_id
            ))
            connection.commit()
            print("✅ Strengths updated from MediaPipe for patient:", patient_user_id)

        else:
            # If data exists, leave as is
            print("ℹ️ Patient already has strength data, skipping update.")

    except Error as e:
        print("❌ Error:", e)

    finally:
        if cursor is not None:
            try:
                cursor.close()
            except Exception:
                pass
        if connection is not None and getattr(connection, "is_connected", lambda: False)():
            try:
                connection.close()
            except Exception:
                pass

# ========================================
# Example usage
# ========================================
# فرضاً قيم MediaPipe اللي أخذتها من الجلسة
media_pipe_values = {
    'shoulder': 15,
    'elbow': 10,
    'wrist': 8,
    'grip': 12,
    'external_rotation': 5
}
#check
update_patient_strengths(patient_user_id=1, **media_pipe_values)

❌ Error: 2003 (HY000): Can't connect to MySQL server on 'localhost:3306' (10061)


: 